KNN

Distância Euclidiana

Manhattan

In [ ]:
import math

def distancia_euclidiana(x, y):
    soma = 0
    for i in range(len(x)):
        soma += (x[i] - y[i]) ** 2
    return math.sqrt(soma)

def distancia_manhattan(x, y):
    soma = 0
    for i in range(len(x)):
        soma += abs(x[i] - y[i])
    return soma

Classificação

In [ ]:
def knn_classificacao(X_train, y_train, x_teste, k, distancia_func):
    distancias = []

    # calcular distância de x_teste para todos
    for i in range(len(X_train)):
        d = distancia_func(x_teste, X_train[i])
        distancias.append((d, y_train[i]))

    # ordenar pela menor distância
    distancias.sort(key=lambda x: x[0])

    # pegar os k mais próximos
    vizinhos = distancias[:k]

    # contar classes
    contagem = {}
    for _, classe in vizinhos:
        if classe not in contagem:
            contagem[classe] = 0
        contagem[classe] += 1

    # retornar classe mais frequente
    return max(contagem, key=contagem.get)

Regressão

In [ ]:
def knn_regressao(X_train, y_train, x_teste, k, distancia_func):
    distancias = []

    for i in range(len(X_train)):
        d = distancia_func(x_teste, X_train[i])
        distancias.append((d, y_train[i]))

    distancias.sort(key=lambda x: x[0])

    vizinhos = distancias[:k]

    # média dos valores
    soma = 0
    for _, valor in vizinhos:
        soma += valor

    return soma / k

pre-processamento

In [1]:
def carregar_arff(caminho):
    X = []
    y = []
    lendo_dados = False

    with open(caminho, 'r') as f:
        for linha in f:
            linha = linha.strip()

            if linha == "" or linha.startswith("%"):
                continue

            if linha.lower() == "@data":
                lendo_dados = True
                continue

            if lendo_dados:
                partes = linha.split(",")

                # última coluna = classe
                atributos = partes[:-1]
                classe = partes[-1]

                # converter números quando possível
                nova_linha = []
                for val in atributos:
                    try:
                        nova_linha.append(float(val))
                    except:
                        nova_linha.append(val)

                X.append(nova_linha)
                y.append(classe)

    return X, y

In [ ]:
def codificar_categoricos(X):
    mapas = [{} for _ in range(len(X[0]))]

    for j in range(len(X[0])):
        valor_id = 0
        for i in range(len(X)):
            val = X[i][j]

            if isinstance(val, str):
                if val not in mapas[j]:
                    mapas[j][val] = valor_id
                    valor_id += 1
                X[i][j] = mapas[j][val]

    return X

In [ ]:
def normalizar(X):
    mins = [min(col) for col in zip(*X)]
    maxs = [max(col) for col in zip(*X)]

    X_norm = []
    for linha in X:
        nova = []
        for i in range(len(linha)):
            if maxs[i] == mins[i]:
                nova.append(0)
            else:
                nova.append((linha[i] - mins[i]) / (maxs[i] - mins[i]))
        X_norm.append(nova)

    return X_norm

In [ ]:
def train_test_split(X, y, proporcao=0.8):
    limite = int(len(X) * proporcao)

    X_train = X[:limite]
    y_train = y[:limite]

    X_test = X[limite:]
    y_test = y[limite:]

    return X_train, X_test, y_train, y_test

In [ ]:
def acuracia(y_real, y_pred):
    acertos = 0
    for i in range(len(y_real)):
        if y_real[i] == y_pred[i]:
            acertos += 1
    return acertos / len(y_real)

In [ ]:
def precision(TP, FP):
    if TP + FP == 0:
        return 0
    return TP / (TP + FP)

In [ ]:
def recall(TP, FN):
    if TP + FN == 0:
        return 0
    return TP / (TP + FN)

In [ ]:
def f1_score(p, r):
    if p + r == 0:
        return 0
    return 2 * (p * r) / (p + r)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

predicoes = []

for x in X_test:
    pred = knn_classificacao(X_train, y_train, x, k=3, distancia_func=distancia_euclidiana)
    predicoes.append(pred)

print("Acurácia:", acuracia(y_test, predicoes))

In [ ]:
def matriz_confusao(y_real, y_pred, positivo="good"):
    TP = FP = TN = FN = 0

    for i in range(len(y_real)):
        if y_real[i] == positivo and y_pred[i] == positivo:
            TP += 1
        elif y_real[i] != positivo and y_pred[i] == positivo:
            FP += 1
        elif y_real[i] != positivo and y_pred[i] != positivo:
            TN += 1
        elif y_real[i] == positivo and y_pred[i] != positivo:
            FN += 1

    return TP, FP, TN, FN

In [ ]:
def calcular_metricas(y_real, y_pred):
    TP, FP, TN, FN = matriz_confusao(y_real, y_pred)

    p = precision(TP, FP)
    r = recall(TP, FN)
    f1 = f1_score(p, r)

    acc = (TP + TN) / (TP + TN + FP + FN)

    return acc, p, r, f1

In [ ]:
def criar_folds(X, y, k):
    tamanho = len(X)
    fold_size = tamanho // k

    folds = []

    for i in range(k):
        inicio = i * fold_size
        fim = inicio + fold_size

        X_test = X[inicio:fim]
        y_test = y[inicio:fim]

        X_train = X[:inicio] + X[fim:]
        y_train = y[:inicio] + y[fim:]

        folds.append((X_train, X_test, y_train, y_test))

    return folds

In [ ]:
def kfold_knn(X, y, k_folds, k_vizinhos, distancia_func):
    folds = criar_folds(X, y, k_folds)

    resultados = []

    for X_train, X_test, y_train, y_test in folds:
        preds = []

        for x in X_test:
            pred = knn_classificacao(X_train, y_train, x, k_vizinhos, distancia_func)
            preds.append(pred)

        acc, p, r, f1 = calcular_metricas(y_test, preds)
        resultados.append((acc, p, r, f1))

    return resultados

In [ ]:
import math

def media(lista):
    return sum(lista) / len(lista)

def desvio_padrao(lista):
    m = media(lista)
    soma = sum((x - m) ** 2 for x in lista)
    return math.sqrt(soma / len(lista))

In [ ]:
def resumo_metricas(resultados):
    accs = [r[0] for r in resultados]
    ps = [r[1] for r in resultados]
    rs = [r[2] for r in resultados]
    f1s = [r[3] for r in resultados]

    print("Accuracy:", media(accs), "±", desvio_padrao(accs))
    print("Precision:", media(ps), "±", desvio_padrao(ps))
    print("Recall:", media(rs), "±", desvio_padrao(rs))
    print("F1:", media(f1s), "±", desvio_padrao(f1s))

In [ ]:
X, y = carregar_arff("datasets/BNG(credit-g)_classificacao.arff")

X = codificar_categoricos(X)
X = normalizar(X)

resultados = kfold_knn(
    X, y,
    k_folds=10,
    k_vizinhos=5,
    distancia_func=distancia_euclidiana
)

resumo_metricas(resultados)